# Evaluate the latest complete Qwen2-VL-2B checkpoint

> **Run this notebook in a separate Colab runtime/session from the training notebook.** It loads a fresh base model and a read-only LoRA checkpoint snapshot, so it does not pause, import, execute, or mutate the training run. Both runtimes may read Google Drive at the same time; the only expected interaction is a small amount of I/O contention during the short checkpoint snapshot.

This notebook:
- finds the newest *fully saved* `checkpoint-N`, falling back if a checkpoint is incomplete, changing, or deleted;
- copies only inference adapter files and JSON/README metadata to this runtime's local SSD;
- evaluates up to all 100 golden examples (60 manifest records plus 20 handwritten and 20 digital manual examples);
- saves an immutable result for the selected checkpoint and updates a lightweight `latest.json` pointer.

Use an A100, L4, or T4 GPU runtime. Run cells in order. The dependency cell performs at most one guarded automatic restart; after reconnecting, rerun from the Drive-mount cell.

In [ ]:
# @title 1) Mount Drive and configure this independent evaluator
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
DRIVE_ROOT = Path("/content/drive/MyDrive/omr")
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "qwen2-vl-2b-handnew"
CHECKPOINT_ROOT = DRIVE_OUTPUT_DIR / "checkpoints"
GOLDEN_ROOT = DRIVE_ROOT / "golden_set"
EVAL_OUTPUT_DIR = DRIVE_OUTPUT_DIR / "midrun-evals"
LOCAL_SNAPSHOT_ROOT = Path("/content/qwen2_vl_checkpoint_snapshots")

MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 768 * 768
EXPECTED_SYNTHETIC = 60
EXPECTED_MANUAL_PER_STYLE = 20
EXPECTED_GOLDEN_TOTAL = 100
CHECKPOINT_STABILITY_WAIT_SECONDS = 5.0
FORCE_RERUN = False  # @param {type:"boolean"}

SYSTEM_PROMPT = "You are an optical music recognition system. Return only PrIMuS semantic tokens."
USER_PROMPT = (
    "Transcribe this monophonic sheet-music image into PrIMuS semantic format. "
    "Return only one space-separated token string with no explanation or markdown."
)

if not CHECKPOINT_ROOT.is_dir():
    raise FileNotFoundError(f"Checkpoint root not found: {CHECKPOINT_ROOT}")
if not GOLDEN_ROOT.is_dir():
    raise FileNotFoundError(f"Golden root not found: {GOLDEN_ROOT}")

LOCAL_SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint root: {CHECKPOINT_ROOT}")
print(f"Golden root:     {GOLDEN_ROOT}")
print(f"Force rerun:     {FORCE_RERUN}")

In [ ]:
# @title 2) Install Qwen2-VL evaluation dependencies (one guarded restart)
# Keep this aligned with the training notebook's Pillow/torchao repair. Do not
# import PIL, transformers, or peft before the repaired runtime has restarted.
import importlib.machinery
import importlib.metadata
import os
import signal
import subprocess
import sys
from pathlib import Path

PILLOW_VERSION = "12.3.0"
_SETUP_REVISION = 3
_SETUP_MARKER = Path(
    f"/content/.qwen2_vl_eval_setup_v{_SETUP_REVISION}_pillow_"
    f"{PILLOW_VERSION.replace('.', '_')}_no_torchao_ready"
)
_RESTART_GUARD = Path(f"/content/.qwen2_vl_eval_setup_v{_SETUP_REVISION}_restart_requested")


def _run_pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", *args])


def _installed_version(distribution: str) -> str | None:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


def _torchao_files_importable() -> bool:
    return importlib.machinery.PathFinder.find_spec("torchao") is not None


def _setup_is_valid() -> bool:
    return (
        _installed_version("pillow") == PILLOW_VERSION
        and _installed_version("torchao") is None
        and not _torchao_files_importable()
        and _installed_version("transformers") is not None
        and _installed_version("accelerate") is not None
        and _installed_version("peft") is not None
        and _installed_version("qwen-vl-utils") is not None
        and _installed_version("safetensors") is not None
    )


if not (_SETUP_MARKER.exists() and _setup_is_valid()):
    if _RESTART_GUARD.exists():
        raise RuntimeError(
            "Dependency setup is still inconsistent after the one guarded restart. "
            "Use Runtime -> Disconnect and delete runtime, then run again. The notebook "
            "will not enter an automatic restart loop."
        )

    print(f"Installing dependencies and a clean Pillow {PILLOW_VERSION} ...", flush=True)
    _run_pip(
        "install",
        "-q",
        "--no-cache-dir",
        "-U",
        "transformers>=4.49.0,<5.0.0",
        "accelerate",
        "peft",
        "datasets",
        "qwen-vl-utils",
        "tqdm",
        "sentencepiece",
        "safetensors",
    )
    # Pillow 12.0.0 can leave mixed PIL modules in Colab. Install the known-good
    # build last, without dependency resolution, so no package can put it back.
    _run_pip(
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "--no-deps",
        f"pillow=={PILLOW_VERSION}",
    )
    # Colab's optional torchao can be incompatible with PEFT. Plain BF16/FP16
    # inference does not use it, so remove it instead of changing PyTorch.
    _run_pip("uninstall", "-q", "-y", "torchao")

    if not _setup_is_valid():
        raise RuntimeError(
            "Dependency repair failed before restart: "
            f"pillow={_installed_version('pillow')}, "
            f"torchao={_installed_version('torchao')}, "
            f"torchao_importable={_torchao_files_importable()}"
        )

    _SETUP_MARKER.write_text(
        f"revision={_SETUP_REVISION}\npillow={PILLOW_VERSION}\ntorchao=absent\n",
        encoding="utf-8",
    )
    _RESTART_GUARD.write_text("automatic restart requested once\n", encoding="utf-8")
    print(
        "Setup complete. Colab will restart once. Reconnect, then rerun from cell 1; "
        "the guard prevents a restart loop.",
        flush=True,
    )
    os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError("Automatic restart failed. Restart the session manually, then rerun from cell 1.")

# Import checks happen only in the clean, restarted process.
import PIL
from PIL import Image, ImageDraw, ImageText
from PIL._typing import _Ink

import accelerate
import peft
import torch
import transformers
from packaging.version import Version

transformers_version = Version(transformers.__version__)
if not (Version("4.49.0") <= transformers_version < Version("5.0.0")):
    raise RuntimeError(f"Expected transformers>=4.49,<5; found {transformers.__version__}")

print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print(f"peft={peft.__version__}")
print(f"accelerate={accelerate.__version__}")
print(f"pillow={PIL.__version__}")
print("torchao=absent; Pillow ImageDraw/ImageText imports: OK")

In [ ]:
# @title 3) Select and snapshot the newest fully written checkpoint
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import tempfile
import time
from pathlib import Path
from typing import Any

CHECKPOINT_PATTERN = re.compile(r"^checkpoint-(\d+)$")
ADAPTER_CONFIG_NAME = "adapter_config.json"
TRAINER_STATE_NAME = "trainer_state.json"
WEIGHT_NAMES = ("adapter_model.safetensors", "adapter_model.bin")
OPTIONAL_METADATA_NAMES = ("README.md",)


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def file_signature(path: Path) -> tuple[int, int]:
    stat = path.stat()
    return stat.st_size, stat.st_mtime_ns


def read_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"Expected a JSON object in {path.name}")
    return payload


def inspect_candidate(checkpoint_dir: Path) -> dict[str, Any]:
    match = CHECKPOINT_PATTERN.fullmatch(checkpoint_dir.name)
    if not match or not checkpoint_dir.is_dir():
        raise ValueError("not a checkpoint-N directory")

    weight_name = next(
        (name for name in WEIGHT_NAMES if (checkpoint_dir / name).is_file()),
        None,
    )
    if weight_name is None:
        raise FileNotFoundError("adapter_model.safetensors or adapter_model.bin is missing")

    required_names = [ADAPTER_CONFIG_NAME, TRAINER_STATE_NAME, weight_name]
    required_paths = [checkpoint_dir / name for name in required_names]
    for path in required_paths:
        if not path.is_file():
            raise FileNotFoundError(f"{path.name} is missing")
        if path.stat().st_size <= 0:
            raise ValueError(f"{path.name} is empty")

    adapter_config = read_json_object(checkpoint_dir / ADAPTER_CONFIG_NAME)
    trainer_state = read_json_object(checkpoint_dir / TRAINER_STATE_NAME)
    copy_names = required_names + [
        name for name in OPTIONAL_METADATA_NAMES if (checkpoint_dir / name).is_file()
    ]
    signatures = {name: file_signature(checkpoint_dir / name) for name in copy_names}
    return {
        "step": int(match.group(1)),
        "weight_name": weight_name,
        "copy_names": copy_names,
        "signatures": signatures,
        "adapter_config": adapter_config,
        "trainer_state": trainer_state,
    }


def snapshot_candidate(checkpoint_dir: Path) -> tuple[Path, dict[str, Any]]:
    first = inspect_candidate(checkpoint_dir)
    print(
        f"Checking checkpoint-{first['step']} for "
        f"{CHECKPOINT_STABILITY_WAIT_SECONDS:.0f}s metadata stability ...",
        flush=True,
    )
    time.sleep(CHECKPOINT_STABILITY_WAIT_SECONDS)
    second = inspect_candidate(checkpoint_dir)
    if (
        first["weight_name"] != second["weight_name"]
        or first["copy_names"] != second["copy_names"]
        or first["signatures"] != second["signatures"]
    ):
        raise RuntimeError("required files changed during the stability window")

    # Hash source files before copying. The local post-copy hashes must match,
    # and Drive metadata must still match after the copy.
    source_hashes = {
        name: sha256_file(checkpoint_dir / name) for name in second["copy_names"]
    }
    temp_dir = Path(tempfile.mkdtemp(prefix=f"checkpoint-{second['step']}-", dir=LOCAL_SNAPSHOT_ROOT))
    try:
        for name in second["copy_names"]:
            shutil.copyfile(checkpoint_dir / name, temp_dir / name)

        local_hashes = {name: sha256_file(temp_dir / name) for name in second["copy_names"]}
        if local_hashes != source_hashes:
            raise RuntimeError("source/local SHA-256 mismatch; checkpoint changed while copying")

        final_signatures = {
            name: file_signature(checkpoint_dir / name) for name in second["copy_names"]
        }
        if final_signatures != second["signatures"]:
            raise RuntimeError("Drive file metadata changed while copying")

        # Parse local JSON again, so the snapshot itself is self-consistent.
        local_adapter_config = read_json_object(temp_dir / ADAPTER_CONFIG_NAME)
        local_trainer_state = read_json_object(temp_dir / TRAINER_STATE_NAME)
        if local_adapter_config != second["adapter_config"]:
            raise RuntimeError("adapter_config.json changed during snapshot")
        if local_trainer_state != second["trainer_state"]:
            raise RuntimeError("trainer_state.json changed during snapshot")

        if second["weight_name"].endswith(".safetensors"):
            from safetensors import safe_open

            with safe_open(temp_dir / second["weight_name"], framework="pt", device="cpu") as handle:
                if not list(handle.keys()):
                    raise ValueError("adapter_model.safetensors contains no tensors")

        final_dir = LOCAL_SNAPSHOT_ROOT / checkpoint_dir.name
        if final_dir.exists():
            shutil.rmtree(final_dir)
        temp_dir.rename(final_dir)
        snapshot_info = {
            **second,
            "source_dir": str(checkpoint_dir),
            "local_dir": str(final_dir),
            "sha256": source_hashes,
            "trainer_epoch": local_trainer_state.get("epoch"),
            "trainer_global_step": local_trainer_state.get("global_step"),
        }
        return final_dir, snapshot_info
    except Exception:
        shutil.rmtree(temp_dir, ignore_errors=True)
        raise


def checkpoint_candidates(root: Path) -> list[Path]:
    candidates: list[tuple[int, Path]] = []
    for path in root.iterdir():
        match = CHECKPOINT_PATTERN.fullmatch(path.name)
        if match and path.is_dir():
            candidates.append((int(match.group(1)), path))
    return [path for _, path in sorted(candidates, key=lambda item: item[0], reverse=True)]


def snapshot_newest_complete_checkpoint(root: Path) -> tuple[Path, dict[str, Any], list[str]]:
    rejected: list[str] = []
    candidates = checkpoint_candidates(root)
    if not candidates:
        raise FileNotFoundError(f"No checkpoint-N directories found under {root}")
    for candidate in candidates:
        try:
            local_dir, info = snapshot_candidate(candidate)
            return local_dir, info, rejected
        except Exception as exc:
            # Any ordinary read/parse/hash/safetensors failure means this candidate
            # is not safe for inference right now; try the next older checkpoint.
            message = f"{candidate.name}: {type(exc).__name__}: {exc}"
            rejected.append(message)
            print(f"Skipping {message}", flush=True)
    raise RuntimeError("No stable, complete checkpoint could be snapshotted:\n" + "\n".join(rejected))


LOCAL_CHECKPOINT_DIR, CHECKPOINT_INFO, REJECTED_CHECKPOINTS = snapshot_newest_complete_checkpoint(
    CHECKPOINT_ROOT
)
SELECTED_CHECKPOINT_STEP = int(CHECKPOINT_INFO["step"])
TRAINER_EPOCH = CHECKPOINT_INFO.get("trainer_epoch")
print(f"Selected checkpoint step: {SELECTED_CHECKPOINT_STEP}")
print(f"Trainer epoch:            {TRAINER_EPOCH if TRAINER_EPOCH is not None else 'unavailable'}")
print(f"Local read-only snapshot: {LOCAL_CHECKPOINT_DIR}")
if REJECTED_CHECKPOINTS:
    print("Newer rejected candidates:")
    for rejection in REJECTED_CHECKPOINTS:
        print(f"  - {rejection}")

In [ ]:
# @title 4) Reuse an existing immutable result unless forced
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CANONICAL_RESULT_PATH = EVAL_OUTPUT_DIR / f"eval_golden_checkpoint-{SELECTED_CHECKPOINT_STEP}.json"
LATEST_RESULT_PATH = EVAL_OUTPUT_DIR / "latest.json"


def print_copy_summary(result: dict, result_path: Path) -> None:
    metrics = result.get("metrics", result)
    checkpoint = result.get("checkpoint", {})
    coverage = result.get("golden_coverage", {})
    n = metrics.get("n_evaluated", metrics.get("n", "?"))
    expected = coverage.get("expected_total", EXPECTED_GOLDEN_TOTAL)
    missing = coverage.get("missing_total", "?")
    exact = metrics.get("exact_matches", "?")
    mean_accuracy = metrics.get("mean_accuracy_pct")
    micro_accuracy = metrics.get("micro_token_accuracy_pct")
    edits = metrics.get("total_token_edits")
    gt_tokens = metrics.get("total_gt_tokens")
    epoch = checkpoint.get("trainer_epoch", TRAINER_EPOCH)

    print("\n=== Qwen2-VL mid-run golden evaluation ===")
    print(f"checkpoint=checkpoint-{SELECTED_CHECKPOINT_STEP}  trainer_epoch={epoch}")
    print(f"golden={n}/{expected}  missing={missing}  exact={exact}/{n}")
    if mean_accuracy is not None:
        print(f"mean_accuracy={mean_accuracy:.2f}%  micro_token_accuracy={micro_accuracy:.2f}%")
    if edits is not None:
        print(f"token_edits={edits}/{gt_tokens}")
    print(f"result={result_path}")


SKIP_EVAL = CANONICAL_RESULT_PATH.is_file() and not FORCE_RERUN
EXISTING_RESULT = None
if SKIP_EVAL:
    try:
        EXISTING_RESULT = json.loads(CANONICAL_RESULT_PATH.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        raise RuntimeError(
            f"Existing immutable result is unreadable: {CANONICAL_RESULT_PATH}. "
            "Inspect it manually; this notebook will not overwrite it."
        ) from exc
    print("Immutable per-checkpoint result already exists; loading it (set FORCE_RERUN=True to rerun).")
    print_copy_summary(EXISTING_RESULT, CANONICAL_RESULT_PATH)
elif CANONICAL_RESULT_PATH.exists():
    print(
        "FORCE_RERUN=True: the canonical result will remain untouched; "
        "this run will be saved as a timestamped immutable rerun artifact."
    )
else:
    print(f"No result exists for checkpoint-{SELECTED_CHECKPOINT_STEP}; evaluation will run.")

In [ ]:
# @title 5) Load a fresh base model and the inference-only LoRA adapter
import gc


def make_messages(target: str | None = None) -> list[dict]:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": USER_PROMPT},
            ],
        },
    ]
    if target is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": target}]})
    return messages


model = None
processor = None
MODEL_DTYPE = None
GPU_NAME = None
if not SKIP_EVAL:
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Colab GPU runtime (A100, L4, or T4) before evaluation.")

    from peft import PeftModel
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

    GPU_NAME = torch.cuda.get_device_name(0)
    MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    dtype_name = "bfloat16" if MODEL_DTYPE == torch.bfloat16 else "float16"
    if not any(name in GPU_NAME.lower() for name in ("a100", "l4", "t4")):
        print(f"Warning: {GPU_NAME} is untested here; continuing with {dtype_name}.")

    adapter_base = CHECKPOINT_INFO["adapter_config"].get("base_model_name_or_path")
    if adapter_base and adapter_base != MODEL_ID:
        raise RuntimeError(
            f"Adapter expects base model {adapter_base!r}, but this evaluator uses {MODEL_ID!r}."
        )

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    print(f"Loading fresh {MODEL_ID} on {GPU_NAME} with {dtype_name} + SDPA ...")
    base_model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=MODEL_DTYPE,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    # This reads adapter config/weights only. Optimizer and scheduler state are
    # intentionally absent from the local snapshot and are never loaded.
    model = PeftModel.from_pretrained(
        base_model,
        str(LOCAL_CHECKPOINT_DIR),
        is_trainable=False,
    )
    model.to("cuda")
    model.requires_grad_(False)
    model.eval()
    model.config.use_cache = True
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Loaded checkpoint-{SELECTED_CHECKPOINT_STEP}; trainable parameters: 0")
else:
    print("Skipping model load because the immutable result was reused.")

In [ ]:
# @title 6) Resolve the 60 synthetic + 40 manual golden records
from collections import Counter, defaultdict

from tqdm.auto import tqdm

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
GROUP_DIRS = {
    "handwritten_original": ("original", "images", "handwritten"),
    "clean_original": ("original", "images", "clean"),
    "camera_distorted_original": ("original", "images", "camera"),
    "handwritten_corrupt": ("corrupt", "images", "handwritten"),
    "clean_corrupt": ("corrupt", "images", "clean"),
    "camera_distorted_corrupt": ("corrupt", "images", "camera"),
}


def normalized_target(path: Path) -> str:
    return " ".join(path.read_text(encoding="utf-8").split())


def find_image(folder: Path, filename_or_stem: str) -> Path | None:
    filename = filename_or_stem.replace("\\", "/").rsplit("/", 1)[-1]
    exact = folder / filename
    if exact.is_file() and exact.suffix.lower() in IMAGE_EXTS:
        return exact
    stem = Path(filename).stem
    return next(
        (
            path
            for path in sorted(folder.glob(f"{stem}.*"))
            if path.is_file() and path.suffix.lower() in IMAGE_EXTS
        ),
        None,
    )


def load_all_golden_records() -> tuple[list[dict], dict]:
    records: list[dict] = []
    seen: set[tuple[str, str]] = set()
    missing_details: list[dict] = []
    duplicates: list[dict] = []
    manifest_path = GOLDEN_ROOT / "manifest.jsonl"
    manifest_rows = 0

    def add_record(record: dict) -> None:
        key = (str(record["group"]), str(record["id"]))
        if key in seen:
            duplicates.append({"group": key[0], "id": key[1]})
            return
        seen.add(key)
        records.append(record)

    if manifest_path.is_file():
        for line_number, line in enumerate(manifest_path.read_text(encoding="utf-8").splitlines(), 1):
            if not line.strip():
                continue
            manifest_rows += 1
            try:
                row = json.loads(line)
                record_id = str(row["id"])
                group = str(row["group"])
                source = str(row.get("source", "original"))
            except (KeyError, TypeError, ValueError, json.JSONDecodeError) as exc:
                missing_details.append(
                    {"kind": "synthetic", "line": line_number, "reason": f"invalid manifest row: {exc}"}
                )
                continue

            group_parts = GROUP_DIRS.get(group)
            if group_parts is None:
                missing_details.append(
                    {"kind": "synthetic", "id": record_id, "group": group, "reason": "unknown group"}
                )
                continue
            image_path = find_image(
                GOLDEN_ROOT.joinpath(*group_parts),
                str(row.get("image_path", record_id)),
            )
            gt_path = GOLDEN_ROOT / source / "ground_truth" / f"{record_id}.semantic"
            if image_path is None or not gt_path.is_file():
                missing_details.append(
                    {
                        "kind": "synthetic",
                        "id": record_id,
                        "group": group,
                        "missing_image": image_path is None,
                        "missing_ground_truth": not gt_path.is_file(),
                    }
                )
                continue
            target = normalized_target(gt_path)
            if not target:
                missing_details.append(
                    {"kind": "synthetic", "id": record_id, "group": group, "reason": "empty ground truth"}
                )
                continue
            add_record(
                {
                    "id": record_id,
                    "group": group,
                    "kind": "synthetic",
                    "image": str(image_path),
                    "target": target,
                }
            )
    else:
        missing_details.append(
            {"kind": "synthetic", "reason": f"manifest missing: {manifest_path}", "expected": EXPECTED_SYNTHETIC}
        )

    manual_gt_dir = GOLDEN_ROOT / "manual" / "ground_truth"
    for style in ("handwritten", "digital"):
        group = f"manual_{style}"
        image_dir = GOLDEN_ROOT / "manual" / style
        for index in range(EXPECTED_MANUAL_PER_STYLE):
            record_id = f"gs{index}"
            image_path = find_image(image_dir, record_id)
            gt_path = manual_gt_dir / f"{record_id}.semantic"
            if image_path is None or not gt_path.is_file():
                missing_details.append(
                    {
                        "kind": "manual",
                        "id": record_id,
                        "group": group,
                        "missing_image": image_path is None,
                        "missing_ground_truth": not gt_path.is_file(),
                    }
                )
                continue
            target = normalized_target(gt_path)
            if not target:
                missing_details.append(
                    {"kind": "manual", "id": record_id, "group": group, "reason": "empty ground truth"}
                )
                continue
            add_record(
                {
                    "id": record_id,
                    "group": group,
                    "kind": "manual",
                    "image": str(image_path),
                    "target": target,
                }
            )

    group_counts = dict(sorted(Counter(record["group"] for record in records).items()))
    coverage = {
        "expected_total": EXPECTED_GOLDEN_TOTAL,
        "expected_synthetic": EXPECTED_SYNTHETIC,
        "expected_manual": 2 * EXPECTED_MANUAL_PER_STYLE,
        "synthetic_manifest_rows": manifest_rows,
        "available_total": len(records),
        "missing_total": max(0, EXPECTED_GOLDEN_TOTAL - len(records)),
        "available_by_group": group_counts,
        "duplicates_skipped": duplicates,
        "missing_details": missing_details,
    }
    return records, coverage


GOLDEN_RECORDS: list[dict] = []
GOLDEN_COVERAGE: dict = {}
if not SKIP_EVAL:
    GOLDEN_RECORDS, GOLDEN_COVERAGE = load_all_golden_records()
    if not GOLDEN_RECORDS:
        raise RuntimeError(f"No usable golden records found under {GOLDEN_ROOT}")
    print(
        f"Golden records available: {len(GOLDEN_RECORDS)}/{EXPECTED_GOLDEN_TOTAL} "
        f"(missing {GOLDEN_COVERAGE['missing_total']})"
    )
    print("By group:", GOLDEN_COVERAGE["available_by_group"])
    if GOLDEN_COVERAGE["synthetic_manifest_rows"] != EXPECTED_SYNTHETIC:
        print(
            f"Warning: manifest has {GOLDEN_COVERAGE['synthetic_manifest_rows']} nonblank rows; "
            f"expected {EXPECTED_SYNTHETIC}."
        )
    if GOLDEN_COVERAGE["missing_total"]:
        print("Manual images may be absent; all available synthetic/manual records will still be evaluated.")
else:
    print("Skipping golden-file scan because the immutable result was reused.")

In [ ]:
# @title 7) Run greedy Qwen inference and project-wide token scoring
import re

_RELATIVE_MINOR_TO_MAJOR = {
    "Am": "CM",
    "Em": "GM",
    "Bm": "DM",
    "F#m": "AM",
    "C#m": "EM",
    "G#m": "BM",
    "D#m": "F#M",
    "A#m": "C#M",
    "Dm": "FM",
    "Gm": "BbM",
    "Cm": "EbM",
    "Fm": "AbM",
    "Bbm": "DbM",
    "Ebm": "GbM",
    "Abm": "CbM",
}
_TIME_SIGNATURE_CANON = {
    "timeSignature-4/4": "timeSignature-C",
    "timeSignature-C": "timeSignature-C",
    "timeSignature-2/2": "timeSignature-C/",
    "timeSignature-C/": "timeSignature-C/",
}


def canonicalize_token(token: str) -> str:
    if token.startswith("keySignature-"):
        key = token[len("keySignature-") :]
        return f"keySignature-{_RELATIVE_MINOR_TO_MAJOR.get(key, key)}"
    return _TIME_SIGNATURE_CANON.get(token, token)


def canonicalize_tokens(tokens: list[str]) -> list[str]:
    return [canonicalize_token(token) for token in tokens]


def token_edit_distance(left: list[str], right: list[str]) -> int:
    previous = list(range(len(right) + 1))
    for i, left_token in enumerate(left, 1):
        current = [i]
        for j, right_token in enumerate(right, 1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[j] + 1,
                    previous[j - 1] + (left_token != right_token),
                )
            )
        previous = current
    return previous[-1]


def strip_one_trailing_barline(tokens: list[str]) -> list[str]:
    return tokens[:-1] if tokens and tokens[-1] == "barline" else tokens


def project_token_distance(prediction_tokens: list[str], target_tokens: list[str]) -> int:
    pred = canonicalize_tokens(prediction_tokens)
    target = canonicalize_tokens(target_tokens)
    stripped_pred = strip_one_trailing_barline(pred)
    stripped_target = strip_one_trailing_barline(target)
    # Forgive at most one trailing barline on either side. Internal barlines and
    # every other discrepancy remain part of the edit distance.
    return min(
        token_edit_distance(pred, target),
        token_edit_distance(stripped_pred, target),
        token_edit_distance(pred, stripped_target),
        token_edit_distance(stripped_pred, stripped_target),
    )


def score_prediction(prediction: str, target: str) -> dict:
    prediction_tokens = prediction.split()
    target_tokens = target.split()
    edits = project_token_distance(prediction_tokens, target_tokens)
    gt_tokens = len(target_tokens)
    if gt_tokens:
        error_rate_pct = edits / gt_tokens * 100.0
        accuracy_pct = max(0.0, 100.0 - error_rate_pct)
    else:
        error_rate_pct = 100.0 if edits else 0.0
        accuracy_pct = 0.0 if edits else 100.0
    return {
        "token_edits": edits,
        "gt_tokens": gt_tokens,
        "prediction_tokens": len(prediction_tokens),
        "error_rate_pct": error_rate_pct,
        "accuracy_pct": accuracy_pct,
        "exact_match": edits == 0,
    }


def clean_prediction(text: str) -> str:
    text = re.sub(r"```(?:\w+)?", "", text or "").replace("```", "")
    return " ".join(text.split())


def predict_one(image_path: str) -> str:
    with Image.open(image_path) as source:
        image = source.convert("RGB")
        image.load()
    prompt_text = processor.apply_chat_template(
        make_messages(),
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = processor(
        text=[prompt_text],
        images=[image],
        padding=True,
        return_tensors="pt",
    ).to(model.device)
    prompt_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )
    generated = output_ids[:, prompt_length:]
    return clean_prediction(
        processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    )


# Executable metric contract checks.
assert project_token_distance(["keySignature-Am"], ["keySignature-CM"]) == 0
assert project_token_distance(["timeSignature-4/4"], ["timeSignature-C"]) == 0
assert project_token_distance(["timeSignature-2/2"], ["timeSignature-C/"]) == 0
assert project_token_distance(["note-C4_quarter", "barline"], ["note-C4_quarter"]) == 0
assert project_token_distance(["barline", "note-C4_quarter"], ["note-C4_quarter"]) == 1
assert score_prediction("a b c d", "x")["accuracy_pct"] == 0.0

EVAL_ROWS: list[dict] = []
if not SKIP_EVAL:
    model.eval()
    for record in tqdm(GOLDEN_RECORDS, desc=f"Golden eval checkpoint-{SELECTED_CHECKPOINT_STEP}"):
        row = {
            "id": record["id"],
            "group": record["group"],
            "kind": record["kind"],
            "image": record["image"],
            "target": record["target"],
        }
        try:
            prediction = predict_one(record["image"])
            row.update({"status": "ok", "prediction": prediction, **score_prediction(prediction, record["target"])})
        except Exception as exc:
            row.update(
                {
                    "status": "error",
                    "prediction": None,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                }
            )
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        EVAL_ROWS.append(row)
    print(f"Inference complete: {sum(row['status'] == 'ok' for row in EVAL_ROWS)} succeeded, "
          f"{sum(row['status'] == 'error' for row in EVAL_ROWS)} failed")
else:
    print("Skipping inference because the immutable result was reused.")

In [ ]:
# @title 8) Save immutable per-checkpoint JSON and print a chat-ready summary
from datetime import datetime, timezone
from uuid import uuid4


def aggregate_rows(rows: list[dict]) -> tuple[dict, dict]:
    successful = [row for row in rows if row.get("status") == "ok"]
    failed = [row for row in rows if row.get("status") == "error"]
    total_edits = sum(int(row["token_edits"]) for row in successful)
    total_gt_tokens = sum(int(row["gt_tokens"]) for row in successful)
    mean_accuracy = (
        sum(float(row["accuracy_pct"]) for row in successful) / len(successful)
        if successful
        else None
    )
    exact_matches = sum(bool(row["exact_match"]) for row in successful)
    if total_gt_tokens:
        micro_accuracy = max(0.0, 100.0 * (1.0 - total_edits / total_gt_tokens))
    else:
        micro_accuracy = 100.0 if successful and total_edits == 0 else 0.0

    overall = {
        "n_available": len(rows),
        "n_evaluated": len(successful),
        "n_inference_errors": len(failed),
        "mean_accuracy_pct": round(mean_accuracy, 4) if mean_accuracy is not None else None,
        "exact_matches": exact_matches,
        "exact_match_pct": round(100.0 * exact_matches / len(successful), 4) if successful else 0.0,
        "total_token_edits": total_edits,
        "total_gt_tokens": total_gt_tokens,
        "micro_token_accuracy_pct": round(micro_accuracy, 4),
    }

    grouped: dict[str, dict] = {}
    all_groups = sorted({row["group"] for row in rows})
    for group in all_groups:
        group_rows = [row for row in rows if row["group"] == group]
        group_ok = [row for row in group_rows if row.get("status") == "ok"]
        group_edits = sum(int(row["token_edits"]) for row in group_ok)
        group_gt = sum(int(row["gt_tokens"]) for row in group_ok)
        group_exact = sum(bool(row["exact_match"]) for row in group_ok)
        group_mean = (
            sum(float(row["accuracy_pct"]) for row in group_ok) / len(group_ok)
            if group_ok
            else None
        )
        group_micro = max(0.0, 100.0 * (1.0 - group_edits / group_gt)) if group_gt else 0.0
        grouped[group] = {
            "n_available": len(group_rows),
            "n_evaluated": len(group_ok),
            "n_inference_errors": len(group_rows) - len(group_ok),
            "mean_accuracy_pct": round(group_mean, 4) if group_mean is not None else None,
            "exact_matches": group_exact,
            "exact_match_pct": round(100.0 * group_exact / len(group_ok), 4) if group_ok else 0.0,
            "total_token_edits": group_edits,
            "total_gt_tokens": group_gt,
            "micro_token_accuracy_pct": round(group_micro, 4),
        }
    return overall, grouped


def write_json_exclusive(path: Path, payload: dict) -> None:
    encoded = (json.dumps(payload, indent=2, ensure_ascii=False) + "\n").encode("utf-8")
    flags = os.O_WRONLY | os.O_CREAT | os.O_EXCL
    fd = os.open(path, flags, 0o644)
    try:
        with os.fdopen(fd, "wb") as handle:
            fd = -1
            handle.write(encoded)
            handle.flush()
            os.fsync(handle.fileno())
    finally:
        if fd >= 0:
            os.close(fd)


def update_latest_pointer(path: Path, payload: dict) -> None:
    temp_path = path.with_name(f".{path.name}.{uuid4().hex}.tmp")
    temp_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    os.replace(temp_path, path)


if not SKIP_EVAL:
    METRICS, PER_GROUP = aggregate_rows(EVAL_ROWS)
    created_at = datetime.now(timezone.utc).isoformat()
    result = {
        "schema_version": 1,
        "created_at_utc": created_at,
        "model_id": MODEL_ID,
        "checkpoint": {
            "step": SELECTED_CHECKPOINT_STEP,
            "name": f"checkpoint-{SELECTED_CHECKPOINT_STEP}",
            "source_dir": CHECKPOINT_INFO["source_dir"],
            "trainer_epoch": TRAINER_EPOCH,
            "trainer_global_step": CHECKPOINT_INFO.get("trainer_global_step"),
            "weight_file": CHECKPOINT_INFO["weight_name"],
            "snapshot_files": CHECKPOINT_INFO["copy_names"],
            "snapshot_sha256": CHECKPOINT_INFO["sha256"],
            "newer_rejected_candidates": REJECTED_CHECKPOINTS,
        },
        "inference": {
            "dtype": "bfloat16" if MODEL_DTYPE == torch.bfloat16 else "float16",
            "gpu": GPU_NAME,
            "attention": "sdpa",
            "max_new_tokens": 256,
            "do_sample": False,
            "num_beams": 1,
            "system_prompt": SYSTEM_PROMPT,
            "user_prompt": USER_PROMPT,
            "processor_min_pixels": MIN_PIXELS,
            "processor_max_pixels": MAX_PIXELS,
        },
        "metric_policy": {
            "unit": "space-separated PrIMuS tokens",
            "relative_major_minor_equivalent": True,
            "time_signature_equivalence": ["4/4 <-> C", "2/2 <-> C/"],
            "trailing_barline": "at most one trailing barline forgiven on either side",
            "accuracy_floor_pct": 0.0,
        },
        "golden_coverage": GOLDEN_COVERAGE,
        "metrics": METRICS,
        "per_group": PER_GROUP,
        "samples": EVAL_ROWS,
    }

    if CANONICAL_RESULT_PATH.exists():
        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        RESULT_PATH = EVAL_OUTPUT_DIR / (
            f"eval_golden_checkpoint-{SELECTED_CHECKPOINT_STEP}_rerun-{timestamp}-{uuid4().hex[:8]}.json"
        )
    else:
        RESULT_PATH = CANONICAL_RESULT_PATH

    # O_EXCL keeps every per-run artifact immutable and prevents accidental
    # overwrite of this or any other checkpoint's result.
    write_json_exclusive(RESULT_PATH, result)
    latest_pointer = {
        "schema_version": 1,
        "updated_at_utc": created_at,
        "checkpoint_step": SELECTED_CHECKPOINT_STEP,
        "trainer_epoch": TRAINER_EPOCH,
        "result_file": RESULT_PATH.name,
        "metrics": METRICS,
        "golden_coverage": {
            "expected_total": GOLDEN_COVERAGE["expected_total"],
            "available_total": GOLDEN_COVERAGE["available_total"],
            "missing_total": GOLDEN_COVERAGE["missing_total"],
        },
    }
    update_latest_pointer(LATEST_RESULT_PATH, latest_pointer)

    print_copy_summary(result, RESULT_PATH)
    print("Per-group mean/micro accuracy:")
    for group, stats in PER_GROUP.items():
        mean_text = "n/a" if stats["mean_accuracy_pct"] is None else f"{stats['mean_accuracy_pct']:.2f}%"
        print(
            f"  {group}: n={stats['n_evaluated']} mean={mean_text} "
            f"micro={stats['micro_token_accuracy_pct']:.2f}% exact={stats['exact_matches']}"
        )
    print(f"latest={LATEST_RESULT_PATH}")
else:
    print("Done: existing per-checkpoint result was left unchanged.")